In [25]:
import pandas as pd
import requests
import json
import os
from datetime import datetime
from dotenv import load_dotenv
from tqdm.notebook import tqdm
from langchain.smith.evaluation import RunEvalConfig

from langsmith import Client
from langchain.smith.evaluation import run_on_dataset
from langsmith.schemas import Run, Example

# Imports para o modo 'local' (exemplo)
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_core.messages import BaseMessage, HumanMessage

print("Bibliotecas importadas.")

Bibliotecas importadas.


In [26]:
# --- MODO DE EXECUÇÃO ---
# Altere para 'api' para testar o serviço Dockerizado
EXECUTION_MODE = 'local'  # Opções: 'local' ou 'api'

# --- CAMINHOS E ENDPOINTS ---
API_URL = "http://localhost:8000/api/v1/atendente"
PERSONAS_CSV_PATH = "data/personas.csv"
DATASET_CSV_PATH = "data/dataset.csv"

# --- CONFIGURAÇÃO LANGSMITH ---
load_dotenv()
PROJECT_NAME_PREFIX = "atendente-workflow-eval"

# --- Carregar dados de contexto ---
personas_df = pd.read_csv(PERSONAS_CSV_PATH).set_index('persona_id')
print(f"Modo de execução: '{EXECUTION_MODE}'")
print(f"{len(personas_df)} personas carregadas.")

Modo de execução: 'local'
2 personas carregadas.


In [27]:
# Exemplo de um workflow LangGraph para ser testado no modo 'local'
# Em um cenário real, você importaria ou definiria seu grafo completo aqui.
class MockAgentState(TypedDict):
    messages: List[BaseMessage]

def mock_agent(state: MockAgentState):
    return {"messages": [HumanMessage(content="Resposta mockada")]}

workflow = StateGraph(MockAgentState)
workflow.add_node("agent", mock_agent)
workflow.set_entry_point("agent")
workflow.add_edge("agent", END)
local_app = workflow.compile()


# --- FUNÇÃO UNIFICADA PARA EXECUTAR O WORKFLOW ---
def run_workflow_target(inputs: dict) -> None:
    """
    Executa o workflow no modo 'local' ou 'api' baseado na configuração.
    Esta função recebe um dicionário de 'inputs' do dataset do LangSmith.
    """
    # 👇 CORREÇÃO: Acessamos 'persona_id' diretamente do dicionário de inputs
    persona_id = inputs['persona_id']
    context = personas_df.loc[persona_id]

    if EXECUTION_MODE == 'local':
        # Injeta o contexto diretamente no input do grafo local
        input_data = {
            "messages": [HumanMessage(content=inputs['user_question'])],
            "context": context.to_dict()
        }
        # A execução é automaticamente traceada pelo LangSmith
        local_app.invoke(input_data)

    elif EXECUTION_MODE == 'api':
        session = requests.Session()
        payload = {"From": "whatsapp:+5521999999999", "Body": inputs['user_question']}
        headers = {"X-VIZU-API-KEY": context['api_key']}
        try:
            response = session.post(API_URL, json=payload, headers=headers, timeout=60)
            response.raise_for_status()
        except requests.RequestException as e:
            print(f"Erro na chamada de API para a persona {persona_id}: {e}")

print("🚀 Função de execução de workflow e grafo local definidos.")

🚀 Função de execução de workflow e grafo local definidos.


In [28]:
def universal_evaluator(run: Run, example: Example) -> dict:
    """
    Avalia a execução do agente inspecionando o trace do LangSmith.

    Ele atua como um roteador: lê a 'golden_answer' do dataset de teste
    e aplica a lógica de avaliação correta, seja para validar uma
    chamada de ferramenta ou o conteúdo de uma resposta em texto.
    """
    try:
        # 1. Carrega a resposta esperada (golden answer) do nosso CSV de teste.
        golden_answer_str = example.outputs.get("golden_answer")
        if not golden_answer_str:
            return {"key": "evaluator_error", "score": 0, "comment": "Golden answer não foi fornecida no dataset."}

        expected_output = json.loads(golden_answer_str)

        # --- ROTA 1: AVALIAR CHAMADA DE FERRAMENTA ---
        if 'tool_name' in expected_output:
            if not run.outputs or 'messages' not in run.outputs or not run.outputs['messages']:
                return {"key": "tool_match", "score": 0, "comment": "FALHA: O trace do Langsmith não contém output de mensagens."}

            last_ai_message = run.outputs['messages'][-1]

            if not hasattr(last_ai_message, 'tool_calls') or not last_ai_message.tool_calls:
                expected_tool = expected_output.get('tool_name')
                return {"key": "tool_match", "score": 0, "comment": f"FALHA: Nenhuma ferramenta foi chamada. Era esperado o uso da ferramenta '{expected_tool}'."}

            # Compara a primeira ferramenta chamada com a esperada
            actual_call = last_ai_message.tool_calls[0]

            tool_name_match = actual_call.get('name') == expected_output.get('tool_name')
            args_match = actual_call.get('args') == expected_output.get('arguments')

            if tool_name_match and args_match:
                return {"key": "tool_match", "score": 1, "comment": "CORRETO: A ferramenta correta foi chamada com os argumentos corretos."}
            else:
                comment = (
                    f"FALHA: Mismatch na chamada da ferramenta.\n"
                    f"-> Esperado: {expected_output.get('tool_name')}({expected_output.get('arguments')})\n"
                    f"-> Recebido:  {actual_call.get('name')}({actual_call.get('args')})"
                )
                return {"key": "tool_match", "score": 0, "comment": comment}

        # --- ROTA 2: AVALIAR CONTEÚDO DA RESPOSTA FINAL ---
        elif 'final_answer_contains' in expected_output:
            if not run.outputs or 'messages' not in run.outputs or not run.outputs['messages']:
                return {"key": "answer_match", "score": 0, "comment": "FALHA: O trace do Langsmith não contém output de mensagens."}

            last_message = run.outputs.get('messages', [])[-1]
            response_text = last_message.content
            expected_text = expected_output['final_answer_contains']

            if expected_text.lower() in response_text.lower():
                return {"key": "answer_match", "score": 1, "comment": "CORRETO: A resposta final contém o texto esperado."}
            else:
                return {"key": "answer_match", "score": 0, "comment": f"FALHA: A resposta não contém '{expected_text}'. Recebido: '{response_text}'"}

        # --- Fallback se o formato da golden_answer for desconhecido ---
        else:
            return {"key": "evaluator_error", "score": 0, "comment": f"Formato de 'golden_answer' não reconhecido: {golden_answer_str}"}

    except Exception as e:
        # Captura qualquer erro inesperado durante a avaliação para facilitar a depuração.
        return {"key": "evaluator_error", "score": 0, "comment": f"Erro irrecuperável no avaliador: {str(e)}"}

print("⚖️ Avaliador 'universal_evaluator' definido com sucesso.")

⚖️ Avaliador 'universal_evaluator' definido com sucesso.


In [31]:
# Imports adicionais para a configuração da avaliação
from langchain.smith.evaluation import RunEvalConfig

# -------------------------------------------------------------------------------------

client = Client()
project_name = f"{PROJECT_NAME_PREFIX}-{datetime.now().strftime('%Y%m%d-%H%M')}"
dataset_name = f"Workflow Evaluation Dataset - {datetime.now().date()}"

# 1. Carregar dataset de teste
eval_df = pd.read_csv(DATASET_CSV_PATH)

# 2. Criar dataset no LangSmith (com verificação manual)
print(f"Garantindo a existência do dataset '{dataset_name}' no LangSmith...")
if client.has_dataset(dataset_name=dataset_name):
    client.delete_dataset(dataset_name=dataset_name)
    print(f"Dataset antigo '{dataset_name}' deletado.")
dataset = client.create_dataset(dataset_name)
print(f"Dataset '{dataset_name}' criado com sucesso.")

for _, row in eval_df.iterrows():
    client.create_example(
        # 👇 CORREÇÃO: Incluímos o 'persona_id' nos inputs
        inputs={
            "user_question": row['input_question'],
            "persona_id": row['client_persona_id']
        },
        outputs={"golden_answer": row['golden_answer']},
        dataset_id=dataset.id,
        metadata={"test_id": row['test_id']} # Podemos manter o test_id aqui
    )
print(f"Dataset preenchido com {len(eval_df)} exemplos.")

# 3. Executar a avaliação
evaluation_config = RunEvalConfig(
    evaluators=[universal_evaluator]
)

print(f"\nIniciando avaliação no modo '{EXECUTION_MODE}'...")
evaluation_run = run_on_dataset(
    client=client,
    dataset_name=dataset_name,
    llm_or_chain_factory=run_workflow_target,
    evaluation=evaluation_config,
    project_name=project_name,
    concurrency_level=5,
    progress=tqdm
)

print(f"\nAVALIAÇÃO CONCLUÍDA! Verifique os resultados no projeto '{project_name}' do LangSmith.")

Garantindo a existência do dataset 'Workflow Evaluation Dataset - 2025-10-07' no LangSmith...
Dataset antigo 'Workflow Evaluation Dataset - 2025-10-07' deletado.
Dataset 'Workflow Evaluation Dataset - 2025-10-07' criado com sucesso.
Dataset preenchido com 3 exemplos.

Iniciando avaliação no modo 'local'...


/Users/lucascruz/Library/Caches/pypoetry/virtualenvs/evaluation-suite-fkzC45bO-py3.11/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3699: LangChainDeprecationWarning: The following arguments are deprecated and will be removed in a future release: dict_keys(['progress']).
  exec(code_obj, self.user_global_ns, self.user_ns)


View the evaluation results for project 'atendente-workflow-eval-20251007-1527' at:
https://smith.langchain.com/o/c7c1bfa6-9c2b-4897-98b2-41e6b537094c/datasets/d56c822e-a990-439c-b908-df9f34651c3d/compare?selectedSessions=9835b4c5-edc2-4648-b425-09d1bba07128

View all tests for Dataset Workflow Evaluation Dataset - 2025-10-07 at:
https://smith.langchain.com/o/c7c1bfa6-9c2b-4897-98b2-41e6b537094c/datasets/d56c822e-a990-439c-b908-df9f34651c3d
[------------------------------------------------->] 3/3
AVALIAÇÃO CONCLUÍDA! Verifique os resultados no projeto 'atendente-workflow-eval-20251007-1527' do LangSmith.


In [32]:
# Puxa os resultados da execução para análise local
results_df = client.get_run_results(run_id=str(evaluation_run.id))

# Exemplo de análise
total_testes = len(results_df)
sucessos = results_df['feedback.tool_match_score'].sum() + results_df['feedback.answer_match_score'].sum()
taxa_sucesso = (sucessos / total_testes) * 100 if total_testes > 0 else 0

print(f"\n--- Relatório de Resultados ---")
print(f"Total de Testes Executados: {total_testes}")
print(f"Sucessos: {int(sucessos)}")
print(f"Taxa de Sucesso: {taxa_sucesso:.2f}%")

# Exibe os 10 piores casos para depuração rápida
print("\nCasos com Falha:")
falhas_df = results_df[~results_df['feedback_level'].str.contains('correct', na=False)]
display(falhas_df[['input.user_question', 'feedback.comment', 'feedback_level']].head(10))

AttributeError: 'Client' object has no attribute 'get_run_results'